<a href="https://colab.research.google.com/github/PabloBenitezR/Aprendizaje-Profundo/blob/main/1_Operaci%C3%B3n_de_convoluci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Operación de convolución
extensión del notebook: https://github.com/gibranfp/CursoAprendizajeProfundo/blob/2026-1/notebooks/2a_convolucion.ipynb, a imágenes a color (múltiples canales).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

from skimage import data
from skimage.transform import resize

## Carga conjunto de datos de dígitos
Vamos a cargar el conjunto de datos de díogitos usando la biblioteca `scikit-learn`:

In [ ]:
digits = load_digits()
data = digits.images / 16 # normalizar
labels = digits.target

In [ ]:
data[0]

Es conveniente normalizar los valores de las imágenes para que estén en el mismo rango ($0-1$).

Este conjuntos de datos está compuesto por imágenes de $8 \times 8$ de los dígitos $0-9$. Visualicemos 2 instancias de los dígitos $0$ y $7$:

In [ ]:
imagen0 = data[labels == 0][0]
imagen1 = data[labels == 1][0]

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(imagen0, cmap = 'gray')
axs[0].set_title('Ejemplo de dígito 0')
axs[1].imshow(imagen1, cmap = 'gray')
axs[1].set_title('Ejemplo de dígito 7')
plt.show()

## Convolución y correlación cruzada
Ahora consideremos las operación de convolución entre una imagen $I$ y un filtro $W$, la cual está definida por

$$
A_{i,j} = (\mathbf{I} * \mathbf{W})_{i,j} = \sum_m \sum_n I_{m, n} W_{i - m, j - n}
$$

La convolución es commutativa, por lo tanto

$$
A_{i,j} = (\mathbf{W} * \mathbf{I})_{i,j} = \sum_m \sum_n I_{i - m, j - n} W_{m,n}
$$

Para este caso con múltiples canales, para una imagen $I\in\mathbb{R}^{H\times W\times C}$ y un filtro $W\in\mathbb{R}^{k\times k\times C}$ la operación es

$$
A_{i,j} = (\mathbf{W} * \mathbf{I})_{i,j} =\sum_{c=1}^C \sum_{m=1}^k \sum_{n=1}^k  I_{i + m, j + n,c} W_{m,n,c}
$$

El resultado de estas operaciones es el mapa de activaciones $A(i,j)$.



In [ ]:
def conv2d_color(I, W, b, stride = 1):
  h, w, c = I.shape
  kh, kw, kc = W.shape
  assert c == kc
  h_s = int(np.floor((h - kh) / stride)) + 1
  w_s = int(np.floor((w - kw) / stride)) + 1
  a = np.zeros((h_s, w_s))

  for i in range(h_s):
    for j in range(w_s):
      I_m = I[i * stride:i * stride + kh, j * stride:j * stride + kw, :]
      a[i, j] = (I_m * W).sum() + b

  return a

## Filtro

Definamos un filtro de $3 \times 3$ que detecte bordes en cierta orientación:

In [ ]:
filtro = np.zeros((3, 3, 3))
np.fill_diagonal(filtro[:, :, 0], 1) #          R
np.fill_diagonal(filtro[:, :, 1], -1) #         G
np.fill_diagonal(filtro[:, :, 2], 0.5) #        B
plt.imshow(filtro * 0.5 + 0.5)
plt.show()


Replicar la imagen en los 3 canales para simular una imagen RGB

In [ ]:
imagen = data[labels == 1][0]
digito_rgb = np.stack([imagen]*3, axis=-1)

activacion = conv2d_color(digito_rgb, filtro, b=0)


fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(digito_rgb)
axs[0].set_title('Imagen RGB')
axs[0].axis('off')
axs[1].imshow(activacion)
axs[1].set_title('Mapa de activacion')
axs[1].axis('off')
plt.show()

Submuestreo maximo (Max Pooling) para imagen RGB

In [ ]:
def max_pooling_color(I, pool_size=(4,4), stride=2):
    h, w, c = I.shape
    ph, pw = pool_size
    h_out = (h - ph) // stride + 1
    w_out = (w - pw) // stride + 1


    pooled = np.zeros((h_out, w_out, c))


    for ch in range(c):
        for i in range(h_out):
            for j in range(w_out):
                patch = I[i*stride:i*stride+ph, j*stride:j*stride+pw, ch]
                pooled[i, j, ch] = np.max(patch)


    return pooled

Aplicar el submuestreo y visualizar el resultado

In [ ]:
imagen_pool = max_pooling_color(digito_rgb, pool_size=(1,1), stride=2)

plt.imshow(imagen_pool)
plt.title('RGB tras submuestreo maximo')
plt.axis('off')
plt.show()